# GD+CITL for Fraunhofer diffraction - analysis trace

Every figure and table in the manuscript regenerates from this notebook against the
immutable results tree. Nothing writes to that tree; nothing reads a hand-edited
intermediate.

Analysis modules live in `code/analysis/` and are versioned with the paper repository.
This notebook is a thin trace over them - the logic lives in the modules, the
sequence lives here.

**Run order:** 0 → 1 → 2 → 3 → … Cells after 2 are independent; 2 produces the
scored dataset they all consume.

## 0 - Configuration

Every path set once. No later cell hardcodes one.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Setup. Nothing here needs editing: the deposit root is found by
# walking up from wherever this notebook happens to be opened, so
# this works on any machine and from any working directory.
# ─────────────────────────────────────────────────────────────
import os, sys, json
from pathlib import Path


def _deposit_root(start=None):
    """First ancestor containing code/analysis/paths.py."""
    p = (start or Path.cwd()).resolve()
    for cand in (p, *p.parents):
        if (cand / "code" / "analysis" / "paths.py").exists():
            return cand
    raise FileNotFoundError(f"could not locate the deposit root from {p}")


ROOT = _deposit_root()
for _sub in ("analysis", "method"):
    _d = str(ROOT / "code" / _sub)
    if _d not in sys.path:
        sys.path.insert(0, _d)

import paths

RESULTS = paths.RESULTS                     # results/
STOCK   = paths.TARGETS                     # targets/
REPLAY  = paths.REPLAY                      # results/experiments/replay_simulation
REPO    = ROOT / "code" / "method"          # where the capture/scoring modules live

FIGS    = paths.FIGURES                     # figures/  (written, not read)
TABLES  = paths.TABLES                      # tables/   (written, not read)
SCORED  = paths.SCORED                      # results/scored.json
ROI     = paths.ROI                         # results/roi.json

# gdcitl reads these, so no module carries a hardcoded path.
os.environ["GDCITL_RESULTS"] = str(RESULTS)
os.environ["GDCITL_REPLAY"]  = str(REPLAY)
os.environ["GDCITL_STOCK"]   = str(STOCK)

for _d in (FIGS, TABLES):
    _d.mkdir(parents=True, exist_ok=True)

# Idempotent: re-running this cell will not warn.
_ip = get_ipython()
if "autoreload" not in _ip.extension_manager.loaded:
    _ip.run_line_magic("load_ext", "autoreload")
_ip.run_line_magic("autoreload", "2")

paths.report()


## 1 - Foundation (`gdcitl.py`)

Shared layer for every figure: loading, terminology, statistics. Defines no plots.

- **Loading** - read-only accessors for each data family.
- **Terminology** - the results JSON stores the no-feedback condition as `naive_*`; the paper calls it *plain GD*. Enforced here so nothing labeled "naive" reaches a caption.
- **Statistics, fixed in advance** - paired Wilcoxon, bootstrap-median CI (seeded, so rebuilds are identical), Benjamini-Hochberg. Medians throughout: the distributions are right-skewed across the target set, so means aren't stable at n = 18.

`check()` inventories every input, including the cross-session calibration match between
the plain-GD captures (24 Jul) and the CITL captures (17 to 20 Jul). **If that reads NO, the
comparison is confounded and nothing downstream is valid.**

In [ ]:
import gdcitl
gdcitl.check()

## 2 - Score all captures

Builds the single scored dataset every results section draws from. Three sources per target per forward model:

| source | what it is |
|---|---|
| `simulation` | GD hologram scored against its own simulated replay |
| `bench_plain_gd` | that same hologram displayed on the SLM and photographed |
| `bench_citl_{random_init,warm_start}` | GD+CITL at its best iteration, photographed |

18 targets × 2 forward models × 4 sources = **144 records**.

**Inputs**
- `results/experiments/{random_init,warm_start}/citl_gs_gd_{ideal,faithful}/` - CITL captures
- `results/experiments/replay_simulation/replay_simulation_*/{ideal,faithful}/<target>/gd.jpg` - plain-GD bench captures, acquired in a separate session
- `results/simulations/simulation_comparison_{ideal,faithful}/<target>_*/outputs/gd/gd_iter_750_recon.npy` - simulated replays
- `targets/` - the eighteen targets

**Output** → `output/scored.json`

**Notes**
- Scoring is `evaluate_methods.compute_metrics`, imported - not reimplemented.
- Metrics are **recomputed**, never read from stored JSON: the saved sweeps predate the resolution-invariant efficiency fix, so 132/198 faithful entries have `diffraction_efficiency = 0.0`.
- Captures are resized to `(H·pad_factor, W·pad_factor)` and `pad_factor` is passed through, so simulation and bench are scored at the **same resolution**. `evaluate_camera_capture_once` does not do this - hence not using it directly.
- Both bench sources get identical zero-order handling (`dc_center` (495,510), `dc_radius` 20, both × `pad_factor`) and identical calibration (manual ROI 3267×3265 @ (1083,129), −0.06°), which is byte-identical across the two capture sessions.

**Run `--dry-run` first** - it inventories all 144 inputs and names anything missing.

*Depends on the setup cell, which puts `code/analysis` and `code/method` on `sys.path`.*

In [ ]:
import score_captures

# dry run first - inventories all 144 inputs, scores nothing
score_captures.score_all(RESULTS, REPLAY, STOCK, REPO, SCORED, dry_run=True)

In [ ]:
# Rebuild scored.json, and report whether anything moved.
#
# The previous contents are read into memory FIRST, so the file written is
# simply scored.json - no second copy to clean up - while the rebuild can
# still be compared against what was there. Without the comparison a rebuild
# that changed a number would look exactly like one that did not.
prev = {(r["target"], r["physics"], r["source"]): r
        for r in json.loads(SCORED.read_text())} if SCORED.exists() else {}

score_captures.score_all(RESULTS, REPLAY, STOCK, REPO, SCORED, dry_run=False)

new = {(r["target"], r["physics"], r["source"]): r
       for r in json.loads(SCORED.read_text())}
METRICS = ("psnr", "ssim", "nmse", "diffraction_efficiency", "lpips", "haarpsi")
# LPIPS and HaarPSI run through a network and a wavelet transform in float32, so
# they can differ in the last digits between a CPU and a GPU. Every other metric
# must match exactly.
TOL = {"lpips": 1e-5, "haarpsi": 1e-5}

if not prev:
    print(f"no previous scored.json to compare against; wrote {len(new)} records")
else:
    print(f"previous {len(prev)} records, rebuilt {len(new)}")
    if set(prev) != set(new):
        print(f"  ** record sets differ: only-previous={len(set(prev)-set(new))}, "
              f"only-new={len(set(new)-set(prev))} **")
    worst, moved = 0.0, 0
    for k in sorted(set(prev) & set(new)):
        for m in METRICS:
            a, b = prev[k].get(m), new[k].get(m)
            if a is None or b is None:
                continue
            if abs(a - b) > TOL.get(m, 1e-9):
                moved += 1
                worst = max(worst, abs(a - b))
                if moved <= 10:
                    print(f"  {'/'.join(k):45s} {m:22s} {a:.6f} -> {b:.6f}")
    print(f"\n{moved} metric(s) moved; largest change {worst:.3e}")
    print("IDENTICAL - this rebuild reproduces the deposited numbers." if moved == 0
          else "** NOT identical. Investigate before depositing. **")


## 3 - GD+CITL vs plain GD on hardware

The result the paper stands on. For each forward model the same 18 targets are photographed
twice on the same bench: once displaying the hologram GD produced in simulation, once
displaying the hologram GD produced with the camera in the loop.

**Within-condition only.** Every comparison holds the forward model fixed and varies only
the hologram - same resolution, same target preparation, same zero-order handling, same
calibration, same normalization. Across conditions nothing is comparable: ideal scores a
native 1000×1000 target, faithful scores that target upsampled to 2000×2000, so the
histograms differ (hence the min-max normalization differs) and SSIM's fixed pixel window
spans different image content. **An ideal PSNR and a faithful PSNR answer two different
scoring problems.**

Ideal vs faithful is therefore a **robustness axis, not a result**: the advantage is
measured twice, independently, and the question is whether it survives both. That is what
answers the fairness objection - GD has no corrector, so its only defense against real
optical non-idealities is the accuracy of its own internal model, and a skeptic is entitled
to ask whether the advantage is an artifact of an unrealistic one.

**Consumes** `scored.json` → **emits** `output/figures/fig_citl_vs_plain.pdf`, `output/tables/citl.tex`, `output/tables/citl_metrics.tex`

In [ ]:
import fig_citl_vs_plain
citl = fig_citl_vs_plain.run(SCORED, FIGS, TABLES)

import numpy as np
for key in citl["order"]:
    c = citl["comparisons"][key]
    print(f"\n=== {c['physics']} / {c['label']} ===")
    for i in np.argsort(c["delta"]):
        print(f"  {c['targets'][i]:16s} plain {c['plain'][i]:6.2f}  "
              f"citl {c['citl'][i]:6.2f}  Δ {c['delta'][i]:+6.2f}")

## 4 - Aberration operating envelope

The bench result shows camera feedback improves reconstructions. It cannot show *why*: a
real optical system's aberration is whatever it happens to be - unknown, and impossible to
sweep. So the bench alone leaves open that the improvement comes from something other than
correcting optical error.

This closes that. We emulate an optical system in simulation, inject an aberration of
**known type and known magnitude**, and withhold it from the optimizer's own forward model -
so the optimizer senses it only through the feedback loop, exactly as it senses a real
bench's aberration only through a real camera. If feedback recovers the target under those
conditions, aberration correction is a mechanism the method genuinely has and has an
explanation rather than just a number.

**Being able to do it in simulation is what makes the experimental result understood rather
than merely observed.** The sweep additionally bounds the mechanism: how much wavefront
error feedback absorbs, and where it stops.

**The comparison.** `aberrated_psnr` (CITL's hologram) vs `naive_aberrated_psnr` (the
no-feedback hologram), *both judged by the system that actually exists*. Scoring CITL's
hologram on the clean model instead would flatter it - that isn't the system it optimized
against.

**The null control.** Piston (Noll *j* = 1) is a pure global phase with exactly zero effect
on replay intensity. Any change measured on piston is optimizer refinement, not correction,
and must be subtracted before a correction magnitude is claimed.

**Reads the Zernike JSONs directly** - this arm is pure simulation and never touches a
capture, so it does not depend on `scored.json`.

**Emits** `output/figures/fig_zernike_envelope.pdf`, `output/figures/fig_zernike_per_mode.pdf` (SI),
`output/tables/zernike_sweep.tex`, `output/tables/zernike_piston.tex`

In [ ]:
import fig_zernike
zern = fig_zernike.run(FIGS, TABLES)

## 5 - Capture cost

Every CITL iteration is one physical camera exposure, so the iteration at which the best
reconstruction occurs *is* the cost of the method - and it's the standing objection to
camera-in-the-loop as a practical technique.

**The claim is a conjunction, and both halves are tested here:**

> **(a)** simulation seeding reaches the best reconstruction in fewer exposures, **and**
> **(b)** it reaches the same quality when it gets there.

(a) alone proves nothing, since any method can stop early if it accepts a worse result.
(b) is what makes the reduction real. The halves come from different measurements
(`summary.log` iteration counts; `scored.json` PSNR), so this cell consumes both.

**Read the interval, not the *p*.** The quality check reports the paired median
difference, seeded minus random, with a bootstrap interval. An interval that sits at or
just above zero supports (b), even when *p* < 0.05, because it means seeding costs no
quality. An interval reaching well below zero would mean it does.

**Emits** `output/figures/fig_capture_cost_{faithful,ideal}.pdf`, `output/tables/capture_cost.tex`

In [ ]:
import fig_capture_cost
cost = fig_capture_cost.run(FIGS, TABLES, scored_path=SCORED)

## 6 - Reconstruction montage

What the improvement actually looks like. Two outputs from **one rendering path**, so the
published figure is provably a subset of what was reviewed:

- `output/figures/reconstructions_{physics}/full/` and `roi/`: every target as a full frame and as a magnified region. Figs. 4 and 5 use seven of these.
- `output/figures/grid/`: the 2×2 panel for every target, for Supplement 1 (next cell).

Three places this figure could quietly mislead, all fixed in code:

**One scale per row.** `vmax` is computed once across *both* the plain-GD and GD+CITL
captures for that target, then applied to both. Per-panel normalization would let either
method be flattered by its own scale, and would make a dim capture look identical to a
bright one. Brightness differences stay visible.

**Zero order excluded from that scale, but shown as recorded.** The display scale is set with the zero-order disc left out, so the spot at the center cannot set the maximum and darken the rest of the frame. The disc itself is not masked in the render, so the zero order appears exactly as the camera recorded it. The metrics are separate: they are computed over the whole frame, zero order included, and section 8 tests whether that matters.

**PSNR, SSIM and LPIPS labels are read from `scored.json`, never recomputed**, so a label cannot disagree
with a table. This is the one figure where numbers are drawn on the image, because here a
number labels the panel it sits on rather than being a statistic read off an axis.

**Choose the montage targets deliberately, not by rank**: a strong case, a typical one, a
weak responder, and the target that goes backwards. A reader who sees the failure case in
the main figure trusts the rest of it more than one who finds it in the supplement.

Loads 36 raw captures (~7 MB each), so allow a minute.

**Consumes** `scored.json` + raw captures → **emits** `output/figures/reconstructions_{physics}/` and `output/figures/grid/`

In [ ]:
import fig_reconstructions

rows = fig_reconstructions.run(SCORED, FIGS, REPO, physics="faithful", roi_file=ROI)

In [ ]:
import fig_reconstructions
grid = fig_reconstructions.run_grid(SCORED, FIGS, REPO)          # all 18

## 7 - Per-iteration convergence

Tables 2 and 3 report where each run *ended* and how many exposures it took to get there.
Neither shows the shape of the descent - and the shape is what the capture-cost claim
actually rests on. Simulation seeding is asserted to buy convergence *speed* rather than
final *quality*; that is a statement about trajectories, and only a trajectory plot can
support or refute it.

**What is plotted, and why it is not PSNR.** The per-iteration quantity that exists is the
**training loss** - the DC-masked mean squared error evaluated on the camera measurement,
i.e. the quantity the optimizer is actually minimizing. Per-iteration PSNR does not exist
and cannot be recovered: each run retained only three captures (iteration 0, the best
iteration, and the last), so there is no reconstruction to score at iteration *k*. **The
caption must say this** - a reader who assumes the y-axis is image quality will misread
the whole figure.

**No normalization.** Loss is plotted raw, on a log axis. Targets differ in achievable MSE
- a dense photograph and a binary chart do not converge to the same floor - so the traces
do not collapse onto one another and the band is correspondingly wide. That is reported
rather than removed: rescaling each target to a common start would hide both the spread in
achievable loss and the seeded run's head start, and the head start is the point.

**Band choice.** Interquartile range across the 18 targets, not a bootstrap interval. This
figure is descriptive - it shows what the runs did. The inferential claims are made in
Tables 2 and 3 with paired tests; an inferential band here would imply a per-iteration
hypothesis test that is not being performed.

**Reads the per-run convergence CSVs directly** - like the aberration sweep, it does not
depend on `scored.json`.

**Emits** `output/figures/fig_convergence.pdf`


In [ ]:
import fig_convergence
conv = fig_convergence.run(FIGS, RESULTS)      # all 4 conditions

## 8 - Zero-order sensitivity

The DC disc is excluded from the training loss and from the capture normalization, but `compute_metrics` scores the full frame, zero order included. A bright zero order could set the maximum used for min-max normalization and compress the rest of the reconstruction, so this section tests whether it does.

This cell rescores all 144 records twice:

- **orig** is `compute_metrics` unchanged. It must reproduce `scored.json`.
- **masked** removes the same disc (radius 20 px native, 40 at P = 2) from both capture and target before normalization. PSNR, MSE and NMSE use the remaining pixels. SSIM averages the map over pixels whose window doesn't touch the disc.

LPIPS and HaarPSI are included. A network can't skip pixels, so for these two the disc is left out of the normalization and filled with the same value in both images.

It then recomputes the four headline paired comparisons, and the seeded-vs-random quality check, under both scorings with the same statistics as sections 3 and 5.

**Decision rule, fixed before running:**

- If every comparison keeps its significance, direction and ordering, the published metric stays primary and the masked one is reported as a sensitivity analysis.
- If any of them changes, the masked metric becomes primary for the bench results.

Read-only. Writes only to `output/dc_diagnostic/radius_<r>/`. Pass a different `radius` to test sensitivity to the mask size.

*Depends on the setup cell.*

In [ ]:
import dc_mask_diagnostic as dcd

dc = dcd.run(radius=20, out=paths.OUTPUT / "dc_diagnostic")

### 8.1 - Mask-radius sensitivity

The radius of 20 px native (40 at P = 2) was chosen by eye to cover the zero-order spot on the bench. It was never tuned against the results. This cell checks that the conclusions don't depend on that choice by rescoring at half the radius, the radius used, and one and a half and two times it.

- **r = 10** leaves the outer part of the spot in, which tests whether a tighter mask would have mattered.
- **r = 20** is the radius used for the loss and the normalization. It should reproduce section 8.
- **r = 30 and r = 40** show the conclusions survive a generous mask. At 40, the disc covers about 0.5% of the frame.

Each capture is loaded once and scored at every radius. The cell writes `output/dc_diagnostic/radius_<r>/` for each radius, plus `radius_sensitivity.txt`, which compares each headline result at r = 0 (the published metric) against every radius. It also writes `output/tables/zero_order.tex`, Table S4 of Supplement 1.

*Depends on the import in section 8.*

In [ ]:
dc_r = dcd.run(radius=[10, 20, 30, 40], out=paths.OUTPUT / "dc_diagnostic")